# 🍜 STEP 04 — MCP 연동: 카카오 지도 API / 법령정보 API

**Model Context Protocol(MCP)**을 사용하면 외부 API를 표준 툴로
에이전트에 연결할 수 있습니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| MCP 개요 | 에이전트 ↔ 외부 시스템 표준 연결 프로토콜 |
| `McpServerPlugin` (SK) | MCP 서버를 SK 플러그인으로 사용 |
| stdio MCP 서버 | 로컬 프로세스로 실행하는 MCP 서버 패턴 |
| API → MCP 래핑 | 카카오 API, 법령 API를 MCP 툴로 변환 |


---
## MCP 개념 이해

```
SK Agent
  └── McpServerPlugin  ←→  MCP Server (stdio/HTTP)
                              ├── 카카오 지도 API 호출
                              ├── 국가법령정보센터 API 호출
                              └── 소상공인 상권정보 API 호출
```

MCP는 LLM 에이전트와 외부 도구 사이의 **표준 통신 프로토콜**입니다.
- **stdio 방식**: 로컬 프로세스로 MCP 서버 실행 (개발·테스트 용이)
- **HTTP 방식**: REST 엔드포인트로 MCP 서버 운영 (프로덕션)

> **F&B 시스템 연결**: LocationAgent의 카카오 지도 API, LegalTaxAgent의 법령 API를 MCP로 연결합니다.

In [ ]:
# MCP 지원을 위한 추가 패키지
%pip install semantic-kernel[mcp] mcp

In [ ]:
import asyncio, os, json
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions import KernelArguments

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview'),
    ))
    return k

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print('임포트 완료')


---
## 1. MCP 서버 직접 구현 — 카카오 지도 API 래핑

> 아래 코드를 `kakao_mcp_server.py`로 저장하면 독립적인 MCP 서버가 됩니다.
> SK 에이전트는 이 서버를 `McpServerPlugin`으로 연결합니다.

In [ ]:
# ✅ [FIX v7] 정확한 URL 방식 확인
# /서비스명/1/N/분기코드 → 분기 전체 수신 후 Python 필터링
# 기본 quarter 값도 실제 데이터 있는 20231로 변경
#
# 실행: python seoul_commercial_mcp_server_sse.py
# ============================================================
FASTMCP_SERVER_CODE = '''import os
import json
import urllib.request
import urllib.parse
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

mcp = FastMCP("seoul-commercial-server")

SEOUL_API_KEY = os.getenv("SEOUL_API_KEY", "")
BASE_URL = "http://openapi.seoul.go.kr:8088"

# 행정동 코드 (서울시 VwsmAdstrdSelngW 실제 데이터 기준으로 검증된 코드)
DONG_CODE_MAP = {
    # 마포구
    "홍대":     "11440660",  # 서교동 (홍대 상권 중심)
    "서교동":   "11440660",
    "홍대입구":  "11440660",
    "연남동":   "11440710",  # 홍대 인근
    "합정":     "11440680",  # 합정동
    "합정동":   "11440680",
    "망원":     "11440690",  # 망원1동
    "망원동":   "11440690",
    "신촌":     "11410530",  # 서대문구 신촌동 (추후 검증 필요)
    # 강남구
    "강남":     "11680640",  # 역삼1동 (추후 검증 필요)
    "역삼":     "11680640",
    # 용산구
    "이태원":   "11170640",  # (추후 검증 필요)
    # 광진구
    "건대":     "11305710",  # (추후 검증 필요)
    # 송파구
    "잠실":     "11710720",  # 잠실7동 (샘플 데이터로 확인됨)
    # 종로구
    "종로":     "11110530",  # (추후 검증 필요)
}

# 업종 코드 (서울시 VwsmAdstrdSelngW 서교동 실제 데이터로 검증된 코드)
INDUSTRY_CODE_MAP = {
    # 음식점
    "한식":       "CS100001",
    "한식음식점":  "CS100001",
    "중식":       "CS100002",
    "일식":       "CS100003",
    "양식":       "CS100004",
    "제과점":     "CS100005",
    "베이커리":    "CS100005",
    "패스트푸드":  "CS100006",
    "치킨":       "CS100007",
    "분식":       "CS100008",
    "호프":       "CS100009",
    "술집":       "CS100009",
    # 카페
    "카페":       "CS100010",  # ✅ 커피-음료 (기존 CS100001 한식 오류 수정)
    "커피":       "CS100010",
    "음료":       "CS100010",
    # 서비스
    "미용실":     "CS200028",
    "네일":       "CS200029",
    "노래방":     "CS200037",
    # 소매
    "편의점":     "CS300002",
    "약국":       "CS300018",
    "화장품":     "CS300022",
}



PAGE_SIZE = 1000  # 서울시 API 최대 페이지 크기


def _fetch_by_quarter(service_name: str, quarter: str, dong_code: str, industry_code: str) -> dict:
    """
    서울시 API 페이지네이션으로 원하는 행정동+업종 데이터 탐색
    - 1000건씩 페이지 요청, 매칭되면 즉시 반환 (불필요한 요청 최소화)
    - 해당 분기 데이터 없으면 최대 4분기 이전까지 자동 fallback
    """
    current_quarter = quarter

    for attempt in range(4):
        # 1단계: 전체 건수 확인
        url_count = "%s/%s/json/%s/1/1/%s" % (
            BASE_URL, urllib.parse.quote(SEOUL_API_KEY), service_name, current_quarter
        )
        safe = url_count.replace(urllib.parse.quote(SEOUL_API_KEY), "***KEY***")
        print("[MCP] 전체건수 확인: %s" % safe)

        with urllib.request.urlopen(url_count, timeout=10) as r:
            data = json.loads(r.read().decode())
        total = data.get(service_name, {}).get("list_total_count", 0)
        print("[MCP] 분기 %s 전체 건수: %d" % (current_quarter, total))

        if total == 0:
            print("[MCP] 데이터 없음 → 이전 분기 fallback")
            current_quarter = _prev_quarter(current_quarter)
            continue

        # 2단계: 1000건씩 페이지네이션으로 탐색
        total_pages = (total + PAGE_SIZE - 1) // PAGE_SIZE
        print("[MCP] 총 %d건 / %d페이지 탐색 시작" % (total, total_pages))

        for page in range(total_pages):
            start    = page * PAGE_SIZE + 1
            end      = min(start + PAGE_SIZE - 1, total)
            url_page = "%s/%s/json/%s/%d/%d/%s" % (
                BASE_URL, urllib.parse.quote(SEOUL_API_KEY),
                service_name, start, end, current_quarter
            )
            safe_page = url_page.replace(urllib.parse.quote(SEOUL_API_KEY), "***KEY***")
            print("[MCP] 페이지 %d/%d: %s" % (page + 1, total_pages, safe_page))

            with urllib.request.urlopen(url_page, timeout=15) as r:
                page_data = json.loads(r.read().decode())
            rows = page_data.get(service_name, {}).get("row", [])

            matched = [
                r for r in rows
                if r.get("ADSTRD_CD") == dong_code
                and r.get("SVC_INDUTY_CD") == industry_code
            ]
            if matched:
                print("[MCP] ✅ 매칭! 페이지 %d에서 발견 (분기: %s)" % (page + 1, current_quarter))
                if current_quarter != quarter:
                    print("[MCP] ⚠️ 요청 분기(%s) 없어 %s 데이터 사용" % (quarter, current_quarter))
                return matched[0]

        print("[MCP] 분기 %s 전체 탐색 완료 - 매칭 없음 → 이전 분기 fallback" % current_quarter)
        current_quarter = _prev_quarter(current_quarter)

    return {}

@mcp.tool()
def get_estimated_sales(location: str, business_type: str, quarter: str = "20253") -> str:
    """
    Query Seoul commercial area estimated monthly sales
    by administrative district and business type (OA-22175 VwsmAdstrdSelngW).
    location: Location in Korean (e.g. 홍대, 강남, 잠실)
    business_type: Business type in Korean (e.g. 카페, 일반음식점)
    quarter: Quarter code YYYYQ. Default is 20253 (latest: 2025 Q3).
      Convert natural language to quarter code before calling:
      - "최신" / no mention          → 20253
      - "1분기"                      → current year + "1" (e.g. 20251)
      - "2024년 2분기" / "24년 2분기" → 20242
      - "작년 3분기"                  → (current_year-1) + "3"
      - "올해 1분기"                  → current_year + "1"
      Always pass a 5-digit string like "20253", never pass natural language.
    """
    dong_code     = DONG_CODE_MAP.get(location, "")
    industry_code = INDUSTRY_CODE_MAP.get(business_type, "CS100001")

    if not SEOUL_API_KEY:
        return json.dumps({"error": "SEOUL_API_KEY not set"}, ensure_ascii=False)
    if not dong_code:
        return json.dumps({"error": "Unsupported location", "supported": list(DONG_CODE_MAP.keys())}, ensure_ascii=False)

    try:
        row = _fetch_by_quarter("VwsmAdstrdSelngW", quarter, dong_code, industry_code)
        if row:
            result = {
                "location":           location,
                "dong_name":          row.get("ADSTRD_CD_NM", ""),
                "business_type":      row.get("SVC_INDUTY_CD_NM", business_type),
                "quarter":            row.get("STDR_YYQU_CD", quarter),
                "monthly_sales_krw":  int(row.get("THSMON_SELNG_AMT", 0)),
                "monthly_tx_count":   int(row.get("THSMON_SELNG_CO", 0)),
                "weekday_sales_krw":  int(row.get("MDWK_SELNG_AMT", 0)),
                "weekend_sales_krw":  int(row.get("WKEND_SELNG_AMT", 0)),
                "time_11_14_krw":     int(row.get("TMZON_11_14_SELNG_AMT", 0)),
                "time_17_21_krw":     int(row.get("TMZON_17_21_SELNG_AMT", 0)),
                "male_sales_krw":     int(row.get("ML_SELNG_AMT", 0)),
                "female_sales_krw":   int(row.get("FML_SELNG_AMT", 0)),
                "age_20s_krw":        int(row.get("AGRDE_20_SELNG_AMT", 0)),
                "age_30s_krw":        int(row.get("AGRDE_30_SELNG_AMT", 0)),
                "age_40s_krw":        int(row.get("AGRDE_40_SELNG_AMT", 0)),
                "source":             "서울시 상권분석서비스 VwsmAdstrdSelngW (OA-22175)",
            }
        else:
            result = {
                "message": "No data",
                "location": location,
                "quarter": quarter,
                "note": "행정동 코드(%s) 또는 업종 코드(%s)가 해당 분기 데이터에 없습니다." % (dong_code, industry_code)
            }
    except Exception as e:
        result = {"error": str(e)}

    return json.dumps(result, ensure_ascii=False)


@mcp.tool()
def get_store_count(location: str, business_type: str, quarter: str = "20253") -> str:
    """
    Query store count and open/close rates
    by administrative district and business type (OA-22172 VwsmAdstrdStorW).
    location: Location in Korean (e.g. 홍대, 강남, 잠실)
    business_type: Business type in Korean (e.g. 카페, 일반음식점)
    quarter: Quarter code YYYYQ. Default is 20253 (latest: 2025 Q3).
      Convert natural language to quarter code before calling:
      - "최신" / no mention          → 20253
      - "1분기"                      → current year + "1" (e.g. 20251)
      - "2024년 2분기" / "24년 2분기" → 20242
      - "작년 3분기"                  → (current_year-1) + "3"
      - "올해 1분기"                  → current_year + "1"
      Always pass a 5-digit string like "20253", never pass natural language.
    """
    dong_code     = DONG_CODE_MAP.get(location, "")
    industry_code = INDUSTRY_CODE_MAP.get(business_type, "CS100001")

    if not SEOUL_API_KEY:
        return json.dumps({"error": "SEOUL_API_KEY not set"}, ensure_ascii=False)
    if not dong_code:
        return json.dumps({"error": "Unsupported location", "supported": list(DONG_CODE_MAP.keys())}, ensure_ascii=False)

    try:
        row = _fetch_by_quarter("VwsmAdstrdStorW", quarter, dong_code, industry_code)
        if row:
            result = {
                "location":        location,
                "dong_name":       row.get("ADSTRD_CD_NM", ""),
                "business_type":   row.get("SVC_INDUTY_CD_NM", business_type),
                "quarter":         row.get("STDR_YYQU_CD", quarter),
                "store_count":     int(row.get("STOR_CO", 0)),
                "open_rate_pct":   float(row.get("OPBIZ_RATE", 0)),
                "close_rate_pct":  float(row.get("CLSBIZ_RATE", 0)),
                "source":          "서울시 상권분석서비스 VwsmAdstrdStorW (OA-22172)",
            }
        else:
            result = {
                "message": "No data",
                "location": location,
                "quarter": quarter,
                "note": "행정동 코드(%s) 또는 업종 코드(%s)가 해당 분기 데이터에 없습니다." % (dong_code, industry_code)
            }
    except Exception as e:
        result = {"error": str(e)}

    return json.dumps(result, ensure_ascii=False)


if __name__ == "__main__":
    import uvicorn
    from mcp.server.sse import SseServerTransport
    from starlette.applications import Starlette
    from starlette.routing import Route, Mount

    port = int(os.getenv("MCP_PORT", "8001"))
    print("Seoul Commercial MCP Server (mcp 1.26 / SSE) on http://localhost:%d/sse" % port)
    print("SEOUL_API_KEY:", "설정됨 ✅" if SEOUL_API_KEY else "❌ 미설정")

    sse_transport = SseServerTransport("/messages/")

    async def handle_sse(request):
        async with sse_transport.connect_sse(
            request.scope, request.receive, request._send
        ) as streams:
            await mcp._mcp_server.run(
                streams[0], streams[1],
                mcp._mcp_server.create_initialization_options()
            )

    starlette_app = Starlette(routes=[
        Route("/sse", endpoint=handle_sse),
        Mount("/messages/", app=sse_transport.handle_post_message),
    ])

    uvicorn.run(starlette_app, host="0.0.0.0", port=port)
'''

with open('seoul_commercial_mcp_server_sse.py', 'w', encoding='utf-8') as f:
    f.write(FASTMCP_SERVER_CODE)

print('seoul_commercial_mcp_server_sse.py 저장 완료! (v7)')
print('터미널: python seoul_commercial_mcp_server_sse.py')


---
## 2. SK에서 MCP 서버 연결 — `McpServerPlugin`

In [ ]:
# ✅ [FIX v3] MCPStdioPlugin(Windows 불가) → MCPSsePlugin(HTTP/SSE)
# 사전 조건: 터미널에서 seoul_commercial_mcp_server_sse.py 실행 중이어야 함
from semantic_kernel.connectors.mcp import MCPSsePlugin

async def create_location_agent_with_mcp() -> tuple:
    """
    Creates a LocationAgent connected to the Seoul Commercial Area MCP server via SSE.
    Prerequisites: seoul_commercial_mcp_server_sse.py must be running on port 8001.
    """
    kernel = make_kernel()

    # SSE 방식: 외부 HTTP 서버에 연결 (subprocess 불필요)
    mcp_plugin = MCPSsePlugin(
        name='SeoulCommercial',
        description='Seoul commercial area analysis using open data API',
        url='http://localhost:8001/sse',
    )
    await mcp_plugin.connect()
    kernel.add_plugin(mcp_plugin)

    agent = ChatCompletionAgent(
        name='LocationAgent',
        description='Commercial area analysis agent using Seoul Open Data API',
        instructions=(
            'You are a commercial area analysis expert for F&B startups in Seoul. '
            'Use get_estimated_sales to retrieve monthly estimated sales data. '
            'Use get_store_count to retrieve competitor store count and open/close rates. '
            'Always call both tools and synthesize results into a Risk / Opportunity analysis. '
            'At the very beginning of your response, always show the data period like: '
            '📅 데이터 기준: YYYY년 N분기 (YYYYQ) '
            'Convert quarter code to Korean: 20253 → 2025년 3분기, 20242 → 2024년 2분기. '
            'Always respond in Korean.'
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_auto_settings()),
    )
    return agent, mcp_plugin

print('SSE 방식 LocationAgent 팩토리 함수 정의 완료')
print('⚠️  seoul_commercial_mcp_server_sse.py 가 실행 중이어야 합니다')
print('    터미널: python seoul_commercial_mcp_server_sse.py')


In [7]:
SEOUL_KEY = os.getenv('SEOUL_API_KEY', '')

if SEOUL_KEY:
    # ── raw JSON 수집용 커널 (에이전트와 별도) ──
    from semantic_kernel.connectors.mcp import MCPSsePlugin

    # async def get_raw_data():
    #     """에이전트 호출 전 MCP 툴 raw 결과를 직접 수집"""
    #     kernel = make_kernel()
    #     plugin = MCPSsePlugin(name='SeoulCommercialRaw', url='http://localhost:8001/sse')
    #     await plugin.connect()
    #     kernel.add_plugin(plugin)
    #     try:
    #         sales = await kernel.invoke(
    #             plugin_name='SeoulCommercialRaw',
    #             function_name='get_estimated_sales',
    #             location='홍대', business_type='카페', quarter='20243',
    #         )
    #         store = await kernel.invoke(
    #             plugin_name='SeoulCommercialRaw',
    #             function_name='get_store_count',
    #             location='홍대', business_type='카페', quarter='20243',
    #         )
    #         return json.loads(str(sales)), json.loads(str(store))
    #     finally:
    #         await plugin.close()

    # # ── 1. raw JSON 먼저 출력 ──
    # sales_raw, store_raw = await get_raw_data()

    # print('=' * 60)
    # print('📦 [RAW] get_estimated_sales')
    # print('=' * 60)
    # print(json.dumps(sales_raw, indent=2, ensure_ascii=False))
    # print()
    # print('=' * 60)
    # print('📦 [RAW] get_store_count')
    # print('=' * 60)
    # print(json.dumps(store_raw, indent=2, ensure_ascii=False))
    # print()

    # ── 2. 에이전트 분석 결과 출력 ──
    location_agent_mcp, mcp_plugin = await create_location_agent_with_mcp()
    try:
        response = await location_agent_mcp.get_response(
            messages=(
                '홍대 상권 분석해줘'
                # 'Analyze the commercial area for a cafe in Hongdae (홍대). '
                # 'Get estimated monthly sales and store count data for Q3 2024. '
                # 'Provide a Risk / Opportunity analysis. Respond in Korean.'
            )
        )
        print('=' * 60)
        print('🤖 [에이전트 분석]')
        print('=' * 60)
        print(response.content)
    finally:
        await mcp_plugin.close()

else:
    print('SEOUL_API_KEY가 설정되지 않았습니다.')
    print('.env 파일에 SEOUL_API_KEY=발급키 를 추가하세요.')


ServiceResponseException: ("<class 'semantic_kernel.connectors.ai.open_ai.services.azure_chat_completion.AzureChatCompletion'> service failed to complete the prompt", AuthenticationError("Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}"))

In [ ]:
# ✅ MCP 툴이 API에서 받아온 raw 데이터 직접 확인
# 방금 에이전트 테스트와 동일한 요청 (홍대, 카페, 2024 Q3)
# ============================================================
from semantic_kernel.connectors.mcp import MCPSsePlugin
from semantic_kernel import Kernel

async def show_raw_mcp_data():
    kernel = make_kernel()

    mcp_plugin = MCPSsePlugin(
        name='SeoulCommercial',
        url='http://localhost:8001/sse',
    )
    await mcp_plugin.connect()
    kernel.add_plugin(mcp_plugin)

    try:
        # ── 툴 1: 추정매출 raw ──
        sales_result = await kernel.invoke(
            plugin_name='SeoulCommercial',
            function_name='get_estimated_sales',
            location='홍대',
            business_type='카페',
            quarter='20243',
        )
        sales_data = json.loads(str(sales_result))

        print('=' * 60)
        print('📊 [RAW] get_estimated_sales — 홍대 카페 2024 Q3')
        print('=' * 60)
        print(json.dumps(sales_data, indent=2, ensure_ascii=False))

        # ── 툴 2: 점포수 raw ──
        store_result = await kernel.invoke(
            plugin_name='SeoulCommercial',
            function_name='get_store_count',
            location='홍대',
            business_type='카페',
            quarter='20243',
        )
        store_data = json.loads(str(store_result))

        print()
        print('=' * 60)
        print('🏪 [RAW] get_store_count — 홍대 카페 2024 Q3')
        print('=' * 60)
        print(json.dumps(store_data, indent=2, ensure_ascii=False))

    finally:
        await mcp_plugin.close()

await show_raw_mcp_data()


---
## 3. 국가법령정보센터 API MCP 서버

> LegalTaxAgent에 연결할 법령 API MCP 서버 코드입니다.

In [ ]:
LAW_MCP_SERVER_CODE = '''
import asyncio
import os
import httpx
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

server = Server("law-info-server")

LAW_API_KEY = os.getenv("LAW_API_KEY", "")  # 국가법령정보센터 발급 키
LAW_BASE = "https://www.law.go.kr/DRF/lawSearch.do"


@server.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="search_law",
            description="국가법령정보센터에서 법령을 검색합니다. 식품위생법, 상가임대차보호법 등 F&B 관련 법령 조회.",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색할 법령명 또는 키워드"},
                },
                "required": ["query"],
            },
        ),
        Tool(
            name="get_law_article",
            description="특정 법령의 조문 내용을 조회합니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "law_name": {"type": "string", "description": "법령명 (예: 식품위생법)"},
                    "article_no": {"type": "string", "description": "조문 번호 (예: 제37조)"},
                },
                "required": ["law_name", "article_no"],
            },
        ),
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    async with httpx.AsyncClient() as client:
        if name == "search_law":
            resp = await client.get(
                LAW_BASE,
                params={
                    "OC": LAW_API_KEY,
                    "target": "law",
                    "type": "JSON",
                    "query": arguments["query"],
                    "display": 5,
                },
            )
            return [TextContent(type="text", text=resp.text)]

    return [TextContent(type="text", text="알 수 없는 툴: %s" % name)]


async def main():
    async with stdio_server() as (read, write):
        await server.run(read, write, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("law_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(LAW_MCP_SERVER_CODE)

print("law_mcp_server.py 저장 완료!")